# 08. AI Analysis: Neural Collapse, Superposition, and WeightWatcher Heavy-Tailed RMT

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Papyan, Han, & Donoho (2020 *Prevalence of neural collapse in deep learning*); Elhage et al. (2022 *Toy Models of Superposition*); Martin & Mahoney (2018–2021 *Implicit Self-Regularization in Deep Neural Networks* / WeightWatcher).
- **Purpose:** Analyze inductive biases and spectral structures in trained networks via Neural Collapse (NC1 within-class variability and NC2 Simplex ETF), Toy Model of Superposition feature interference, and WeightWatcher heavy-tailed RMT tail exponents.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [07. AI Analysis Representation Similarity CKA RSA](07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.ipynb) $\to$ `🔗 Downstream Syntheses:` [09. AI Analysis Loss Landscape Hessian Lanczos and Empirical NTK](09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not apply Simplex ETF metrics to unnormalized features or severe class-imbalanced datasets where geometry is naturally asymmetric; avoid setting Hill estimator $\lambda_{\min}$ inside the Marchenko-Pastur bulk, which corrupts power-law fits with random noise.

## Mathematical Foundations & Governing Equations

1. **NC1 (Within-Class Variability)**: $\text{NC1} = \frac{1}{N}\sum_{c=1}^C \sum_{i=1}^{N_c} \|h_{i, c} - \mu_c\|_2^2 \to 0$
2. **NC2 (Simplex ETF Cosine Error)**: $\mathcal{NC}2_{\text{err}} = \frac{1}{C(C-1)}\sum_{c \neq c'} \left(\langle \tilde{\mu}_c, \tilde{\mu}_{c'}\rangle - \left(-\frac{1}{C-1}\right)\right)^2$
3. **Superposition Interference**: $G = W^T W, \quad \mathcal{E}_{\text{interf}} = \sum_{i \neq j} G_{ij}^2$
4. **WeightWatcher Hill Estimator**: $\hat{\alpha} = 1 + \frac{K}{\sum_{k=1}^K \ln(\lambda_k / \lambda_{\min})}$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_nc(feats, labels, num_classes=3):
    n, d = feats.shape
    c = num_classes
    means = torch.zeros(c, d)
    counts = torch.zeros(c)
    for i in range(n):
        l = int(labels[i].item())
        means[l] += feats[i]
        counts[l] += 1.0
    for k in range(c):
        if counts[k] > 0: means[k] /= counts[k]
    nc1 = sum(((feats[i] - means[int(labels[i].item())])**2).sum() for i in range(n)) / float(n)
    g_mean = means.mean(dim=0, keepdim=True)
    c_means = means - g_mean
    norms = torch.sqrt((c_means * c_means).sum(dim=-1, keepdim=True) + 1e-8)
    normed = c_means / norms
    cos_mat = normed @ normed.t()
    ideal_cos = -1.0 / float(c - 1)
    nc2 = sum((cos_mat[i, j] - ideal_cos)**2 for i in range(c) for j in range(c) if i != j) / float(c * (c - 1))
    return nc1, nc2

def calc_superposition(w):
    gram = w.t() @ w
    diag = torch.diag(torch.diag(gram))
    off_diag = gram - diag
    return gram, (off_diag * off_diag).sum()

def calc_ww_alpha(evals, lambda_min=0.5):
    tail = evals[evals >= lambda_min]
    k = float(tail.shape[0])
    return 1.0 + k / (torch.log(tail / lambda_min).sum() + 1e-8)

In [ ]:
# Parity verification against golden reference tensors
nc1, nc2 = calc_nc(weights["nc_feats"], weights["nc_labels"], num_classes=3)
assert torch.max(torch.abs(nc1 - weights["nc1_ref"])) < 1e-5, "NC1 mismatch"
assert torch.max(torch.abs(nc2 - weights["nc2_ref"])) < 1e-5, "NC2 mismatch"

gram, interf = calc_superposition(weights["sp_w"])
assert torch.max(torch.abs(gram - weights["sp_gram_ref"])) < 1e-5, "Gram mismatch"
assert torch.max(torch.abs(interf - weights["sp_interf_ref"])) < 1e-5, "Interference mismatch"

alpha = calc_ww_alpha(weights["ww_evals"], lambda_min=0.5)
assert torch.max(torch.abs(alpha - weights["ww_alpha_ref"])) < 1e-5, "WeightWatcher alpha mismatch"
print("All Neural Collapse, Superposition, and WeightWatcher assertions passed!")

In [ ]:
# Benchmarking analysis metrics
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_nc(weights["nc_feats"], weights["nc_labels"], num_classes=3)
    _ = calc_superposition(weights["sp_w"])
    _ = calc_ww_alpha(weights["ww_evals"], lambda_min=0.5)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused NC + Superposition + WeightWatcher latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Dense vs Sparse Superposition Cross-Talk with ReLU thresholding
# 4 features packed into 2 dimensions
w = weights["sp_w"]  # [2, 4]
threshold = 0.75
# Sparse activation: only feature 0 active
x_sparse = torch.tensor([1.0, 0.0, 0.0, 0.0])
rec_sparse = torch.relu(w.t() @ (w @ x_sparse) - threshold)
err_sparse = (rec_sparse[1:]**2).sum().item()  # interference cleanly filtered

# Dense activation: all 4 features active simultaneously
x_dense = torch.tensor([1.0, 1.0, 1.0, 1.0])
rec_dense = torch.relu(w.t() @ (w @ x_dense) - threshold)
err_dense = ((rec_dense - x_dense)**2).sum().item()
print(f"Sparse feature cross-talk: {err_sparse:.4f} vs Dense cross-talk: {err_dense:.4f}")
assert err_sparse < 0.01 and err_dense > 1.0, "Superposition requires sparsity to cleanly separate features!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Neural Collapse** demonstrates that terminal-phase training compresses within-class variance (NC1) and aligns class means into a Simplex Equiangular Tight Frame (NC2).
2. **Superposition** allows neural networks to represent more features than dimensions ($M > D$) provided features are sufficiently sparse to bound interference.
3. **WeightWatcher Heavy-Tailed RMT** measures implicit self-regularization via power-law tail exponent $\alpha \in [2, 4]$ without requiring test sets.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.md](../../markdown/10_xai_and_ai_analysis_methods/08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.md)
- **PyTorch 2.x Module:** [08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.py](../../pytorch2/10_xai_and_ai_analysis_methods/08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.py)
- **Functional JAX Module:** [08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.py](../../jax/10_xai_and_ai_analysis_methods/08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.py)
- **Native Rust Candle Track:** [08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.rs](../../candle/10_xai_and_ai_analysis_methods/08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.rs)
- **Hardware Kernel (CPU / WGSL):** [08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.rs](../../kernels/10_xai_and_ai_analysis_methods/08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.rs)
- **Golden SafeTensors:** [08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.safetensors](../../weights/10_xai_and_ai_analysis_methods/08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 07. AI Analysis Representation Similarity CKA RSA](07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 09. AI Analysis Loss Landscape Hessian Lanczos and Empirical NTK](09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.ipynb) &rarr;